# الفصل 22: حفظ نقاط التفتيش ومراقبة التدريب

> طبقة الحدس تخيّل أنك تُدرِّب نموذجًا لمدة 3 أسابيع متواصلة على 512 GPU (تكلفة تقديرية قد تتجاوز مئات الآلاف من الدولارات).

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 22
- **المستوى:** متقدم
- **زمن القراءة:** 25 دقيقة
- **المتطلبات:** الفصل 21: التوازي الموزّع للتدريب
- **الوسوم:** Checkpointing، Observability، Fault Tolerance، Cost

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, torch, wandb
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("wandb", "wandb")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch22-checkpointing-monitoring.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس تخيّل أنك تُدرِّب نموذجًا لمدة 3 أسابيع متواصلة على 512 GPU (تكلفة تقديرية قد تتجاوز مئات الآلاف من الدولارات).

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** متقدم
> - **زمن القراءة:** 25 دقيقة
> - **المتطلبات:** الفصل 21: التوازي الموزّع للتدريب


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم ماذا يجب أن يحتوي Checkpoint كامل؟
- فهم Distributed Checkpointing مع ZeRO/FSDP
- فهم مراقبة التدريب: المقاييس الأساسية


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

تخيّل أنك تُدرِّب نموذجًا لمدة **3 أسابيع متواصلة** على 512 GPU (تكلفة تقديرية قد تتجاوز مئات الآلاف من الدولارات). في اليوم الثامن عشر، يحدث عطل في إحدى العقد (شائع جدًا في الأنظمة الموزَّعة الضخمة؛ احتمال عطل جزء من الأجهزة يزداد مع زيادة عدد الأجزاء). بدون نظام Checkpointing سليم، ستفقد **18 يومًا من التدريب بالكامل**!

بالإضافة لذلك، تدريب LLM عملية **طويلة الأمد وغامضة جزئيًا**: الخسارة قد تنخفض بسلاسة لأيام، ثم فجأة "تنفجر" (Loss Spike) لأسباب متنوعة (بيانات شاذة، عدم استقرار رقمي، معدل تعلم مرتفع جدًا لحظيًا). بدون مراقبة دقيقة ومستمرة، قد لا تكتشف المشكلة إلا بعد ساعات أو أيام من إهدار الحوسبة على تدريب فاسد.

هذا الفصل يغطي البنية التحتية الأساسية (لكن الحيوية) لأي تدريب LLM جاد: كيف تحفظ حالة التدريب بأمان، وكيف تراقبه بذكاء لاكتشاف المشاكل مبكرًا.


## 2. الخريطة الذهنية


```text
البنية التحتية للتدريب الموثوق
│
├── 1. Checkpointing (حفظ واستعادة حالة التدريب)
│     - ماذا نحفظ؟ (أوزان + حالات محسِّن + جدول LR + خطوة التدريب + RNG state)
│     - متى نحفظ؟ (كل N خطوة، تريد-أوف بين التكلفة والأمان)
│     - Distributed Checkpointing (حفظ موزَّع متوافق مع ZeRO/FSDP)
│
├── 2. مراقبة التدريب (Training Monitoring)
│     - مقاييس أساسية: Loss, Learning Rate, Gradient Norm
│     - مقاييس متقدمة: Loss Spikes Detection, Activation Statistics
│     - أدوات: Weights & Biases (wandb), TensorBoard, MLflow
│
├── 3. اكتشاف ومعالجة Loss Spikes
│     - أسباب شائعة: بيانات شاذة، Numerical Instability، LR مرتفع
│     - استراتيجيات التعافي: Gradient Clipping، إعادة التحميل من checkpoint سابق
│
└── 4. تقدير تكلفة التدريب
      GPU-hours, التكلفة بالدولار، مقارنة سيناريوهات مختلفة
```


## 3. الشرح المتدرج

### 3.1 ماذا يجب أن يحتوي Checkpoint كامل؟

> 🟡 **طبقة الآلية**

خطأ شائع جدًا للمبتدئين: الاعتقاد أن حفظ **أوزان النموذج فقط** كافٍ لاستئناف التدريب. في الحقيقة، Checkpoint كامل وموثوق يجب أن يحتوي:


In [ ]:
import torch
import os
import json
import random
import numpy as np
from datetime import datetime

class CheckpointManager:
    """
    مدير Checkpoints كامل للتدريب طويل الأمد، يضمن إمكانية استئناف
    التدريب من أي نقطة توقف بشكل مطابق تمامًا (Deterministic Resume)
    لما كان سيحدث لو لم يتوقف التدريب إطلاقًا.
    """
    def __init__(self, checkpoint_dir: str, keep_last_n: int = 3):
        self.checkpoint_dir = checkpoint_dir
        self.keep_last_n = keep_last_n
        os.makedirs(checkpoint_dir, exist_ok=True)

    def save_checkpoint(self,
                         model: torch.nn.Module,
                         optimizer: torch.optim.Optimizer,
                         lr_scheduler,
                         step: int,
                         epoch: int,
                         loss: float,
                         extra_metadata: dict = None):
        """
        حفظ Checkpoint كامل يحتوي كل ما يلزم لاستئناف التدريب
        بشكل مطابق تمامًا من حيث الحالة الحسابية.
        """
        checkpoint = {
            # 1. أوزان النموذج - العنصر الأوضح لكن غير الكافي وحده
            "model_state_dict": model.state_dict(),

            # 2. حالات المُحسِّن - حرجة جدًا! بدونها، Adam/AdamW يفقد
            # متوسطاته المتحركة (m, v) المتراكمة ويبدأ التحسين "من الصفر"
            # فعليًا حتى لو كانت الأوزان محفوظة، مما يسبب اضطرابًا كبيرًا
            # في منحنى الخسارة عند الاستئناف
            "optimizer_state_dict": optimizer.state_dict(),

            # 3. حالة جدول معدل التعلم (مهم لمعرفة أين نحن في Warmup/Decay)
            "lr_scheduler_state_dict": lr_scheduler.state_dict() if lr_scheduler else None,

            # 4. عداد الخطوات/العصور - لاستئناف جدول LR وDataLoader بشكل صحيح
            "step": step,
            "epoch": epoch,

            # 5. قيمة الخسارة عند هذه النقطة - للمقارنة والتحقق من السلامة
            "loss": loss,

            # 6. حالة مولدات الأرقام العشوائية - لضمان قابلية إعادة الإنتاج
            # الكاملة (نفس تسلسل Dropout/أي عشوائية أخرى عند الاستئناف)
            "rng_state": {
                "torch": torch.get_rng_state(),
                "torch_cuda": torch.cuda.get_rng_state_all()
                              if torch.cuda.is_available() else None,
                "numpy": np.random.get_state(),
                "python": random.getstate(),
            },

            # 7. بيانات وصفية إضافية (تاريخ الحفظ، إصدار الكود، إعدادات التدريب)
            "metadata": {
                "timestamp": datetime.now().isoformat(),
                **(extra_metadata or {}),
            },
        }

        checkpoint_path = os.path.join(
            self.checkpoint_dir, f"checkpoint_step_{step}.pt"
        )
        # الحفظ بشكل ذري (atomic): نكتب لملف مؤقت أولاً ثم نُعيد تسميته،
        # لتفادي ملف checkpoint تالف إذا انقطع التيار أثناء الكتابة
        tmp_path = checkpoint_path + ".tmp"
        torch.save(checkpoint, tmp_path)
        os.rename(tmp_path, checkpoint_path)

        print(f"✅ تم حفظ Checkpoint عند الخطوة {step} في {checkpoint_path}")
        self._cleanup_old_checkpoints()

    def _cleanup_old_checkpoints(self):
        """الاحتفاظ فقط بآخر N نسخة لتوفير مساحة التخزين (checkpoints ضخمة جدًا)"""
        checkpoints = sorted([
            f for f in os.listdir(self.checkpoint_dir)
            if f.startswith("checkpoint_step_")
        ], key=lambda x: int(x.split("_")[-1].replace(".pt", "")))

        while len(checkpoints) > self.keep_last_n:
            oldest = checkpoints.pop(0)
            os.remove(os.path.join(self.checkpoint_dir, oldest))
            print(f"🗑️  حُذف Checkpoint قديم: {oldest}")

    def load_checkpoint(self, checkpoint_path: str, model, optimizer, lr_scheduler):
        """استعادة كاملة لحالة التدريب من Checkpoint محفوظ"""
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if lr_scheduler and checkpoint["lr_scheduler_state_dict"]:
            lr_scheduler.load_state_dict(checkpoint["lr_scheduler_state_dict"])

        # استعادة حالة العشوائية لضمان الاستمرارية المطابقة
        rng_state = checkpoint["rng_state"]
        torch.set_rng_state(rng_state["torch"])
        if rng_state["torch_cuda"] is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(rng_state["torch_cuda"])
        np.random.set_state(rng_state["numpy"])
        random.setstate(rng_state["python"])

        print(f"✅ تم استعادة Checkpoint من الخطوة {checkpoint['step']}، "
              f"الخسارة عند الحفظ: {checkpoint['loss']:.4f}")

        return checkpoint["step"], checkpoint["epoch"]

    def find_latest_checkpoint(self) -> str:
        """إيجاد أحدث Checkpoint متاح تلقائيًا (لاستئناف تلقائي بعد انقطاع)"""
        checkpoints = [
            f for f in os.listdir(self.checkpoint_dir)
            if f.startswith("checkpoint_step_")
        ]
        if not checkpoints:
            return None
        latest = max(checkpoints,
                      key=lambda x: int(x.split("_")[-1].replace(".pt", "")))
        return os.path.join(self.checkpoint_dir, latest)


> ⚠️ **خطأ شائع خطير**: نسيان حفظ **حالات المُحسِّن (Optimizer States)**. حفظ الأوزان فقط ثم استئناف التدريب بمُحسِّن "جديد" (بدون متوسطاته المتحركة m وv المتراكمة) يُسبب غالبًا **قفزة مفاجئة في الخسارة (Loss Spike)** عند الاستئناف، لأن AdamW يفقد "ذاكرته" عن اتجاه وتذبذب التدرجات السابقة ويحتاج عدة خطوات لإعادة "الإحماء" من جديد.

### 3.2 Distributed Checkpointing مع ZeRO/FSDP

عندما تكون الأوزان مُقسَّمة (Sharded) عبر عشرات/مئات الـ GPUs (كما في ZeRO-3/FSDP)، حفظ Checkpoint يصبح أكثر تعقيدًا: إما تجميع كل الشرائح في GPU واحد قبل الحفظ (مكلف ذاكريًا لحظيًا)، أو حفظ كل شريحة بشكل منفصل (Sharded Checkpoint).


In [ ]:
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import StateDictType, FullStateDictConfig

def save_fsdp_checkpoint_consolidated(fsdp_model: FSDP, save_path: str, rank: int):
    """
    حفظ Checkpoint مُجمَّع (Consolidated) من نموذج FSDP: يُجمِّع كل الشرائح
    المُوزَّعة في state_dict واحد كامل على rank=0 فقط، مناسب للتوزيع
    اللاحق (نشر النموذج، أو تحميله لاحقًا على عدد GPUs مختلف).

    العيب: يتطلب ذاكرة كافية على rank=0 لاستيعاب النموذج الكامل لحظيًا.
    """
    full_state_dict_config = FullStateDictConfig(
        offload_to_cpu=True,  # نقل الحالة المُجمَّعة لذاكرة CPU لتفادي OOM على GPU
        rank0_only=True,       # فقط rank=0 يحتفظ بالنسخة الكاملة النهائية
    )

    with FSDP.state_dict_type(
        fsdp_model, StateDictType.FULL_STATE_DICT, full_state_dict_config
    ):
        full_state_dict = fsdp_model.state_dict()

    if rank == 0:
        torch.save(full_state_dict, save_path)
        print(f"✅ تم حفظ Checkpoint مُجمَّع كامل في {save_path}")

def save_fsdp_checkpoint_sharded(fsdp_model: FSDP, save_dir: str):
    """
    حفظ Checkpoint مُجزَّأ (Sharded): كل GPU يحفظ شريحته الخاصة فقط
    في ملف منفصل. أسرع بكثير وأقل استهلاكًا للذاكرة من النهج المُجمَّع،
    وهو الخيار المُفضَّل للـ checkpoints الدورية أثناء التدريب (الاستئناف
    يكون غالبًا على نفس عدد GPUs بالضبط).
    """
    from torch.distributed.fsdp import ShardedStateDictConfig

    with FSDP.state_dict_type(
        fsdp_model, StateDictType.SHARDED_STATE_DICT,
        ShardedStateDictConfig(offload_to_cpu=True)
    ):
        sharded_state_dict = fsdp_model.state_dict()
        rank = dist.get_rank()
        torch.save(sharded_state_dict,
                   os.path.join(save_dir, f"shard_rank_{rank}.pt"))


> 💡 **نصيحة عملية**: استخدم **Sharded Checkpointing** للحفظ الدوري المتكرر أثناء التدريب (أسرع وأخف)، واحفظ **Checkpoint مُجمَّع كامل** فقط عند نقاط مهمة (نهاية التدريب، قبل مرحلة تقييم رئيسية، أو عند تغيير عدد GPUs المستخدَمة لاحقًا).

### 3.3 مراقبة التدريب: المقاييس الأساسية

> 🔴 **طبقة التعمّق**


In [ ]:
import torch
import time
from collections import deque

class TrainingMonitor:
    """
    نظام مراقبة شامل للتدريب يتتبع المقاييس الحيوية ويكتشف المشاكل
    مبكرًا قبل أن تُهدر ساعات/أيام من الحوسبة على تدريب فاسد.
    """
    def __init__(self, window_size: int = 100,
                 loss_spike_threshold: float = 3.0,
                 grad_norm_explosion_threshold: float = 100.0):
        self.window_size = window_size
        self.loss_spike_threshold = loss_spike_threshold
        self.grad_norm_explosion_threshold = grad_norm_explosion_threshold

        self.loss_history = deque(maxlen=window_size)
        self.grad_norm_history = deque(maxlen=window_size)
        self.step_times = deque(maxlen=window_size)
        self.alerts_log = []

    def compute_gradient_norm(self, model: torch.nn.Module) -> float:
        """
        حساب القاعدة الإقليدية الكلية (L2 norm) لكل تدرجات النموذج،
        وهو مقياس أساسي لاستقرار التدريب. قيم مرتفعة جدًا فجأة تشير
        لعدم استقرار رقمي ("Gradient Explosion").
        """
        total_norm_squared = 0.0
        for param in model.parameters():
            if param.grad is not None:
                param_norm = param.grad.data.norm(2)
                total_norm_squared += param_norm.item() ** 2
        return total_norm_squared ** 0.5

    def log_step(self, step: int, loss: float, model: torch.nn.Module,
                 learning_rate: float, step_duration: float) -> dict:
        """
        تسجيل مقاييس خطوة تدريب واحدة، مع كشف تلقائي عن الحالات الشاذة.
        يُعيد قاموسًا بأي تنبيهات اكتُشفت (فارغ إذا كل شيء طبيعي).
        """
        grad_norm = self.compute_gradient_norm(model)
        alerts = {}

        # 1. كشف Loss Spike: الخسارة الحالية أعلى بكثير من المتوسط المتحرك الأخير
        if len(self.loss_history) >= 10:
            recent_mean = sum(list(self.loss_history)[-10:]) / 10
            recent_std = (sum((x - recent_mean)**2
                          for x in list(self.loss_history)[-10:]) / 10) ** 0.5
            if recent_std > 0 and loss > recent_mean + self.loss_spike_threshold * recent_std:
                alerts["loss_spike"] = (
                    f"⚠️ Loss Spike عند الخطوة {step}: "
                    f"{loss:.4f} (المتوسط الأخير: {recent_mean:.4f} ± {recent_std:.4f})"
                )

        # 2. كشف Gradient Explosion
        if grad_norm > self.grad_norm_explosion_threshold:
            alerts["grad_explosion"] = (
                f"⚠️ Gradient Norm مرتفع جدًا عند الخطوة {step}: {grad_norm:.2f}"
            )

        # 3. كشف NaN/Inf (الأخطر على الإطلاق - يُفسد التدريب فورًا)
        if torch.isnan(torch.tensor(loss)) or torch.isinf(torch.tensor(loss)):
            alerts["nan_inf_loss"] = (
                f"🔴 خطأ حرج: الخسارة NaN/Inf عند الخطوة {step}! "
                f"يجب إيقاف التدريب فورًا والاستعادة من آخر checkpoint سليم"
            )

        # تحديث السجلات
        self.loss_history.append(loss)
        self.grad_norm_history.append(grad_norm)
        self.step_times.append(step_duration)

        if alerts:
            self.alerts_log.extend(alerts.values())
            for msg in alerts.values():
                print(msg)

        return {
            "step": step,
            "loss": loss,
            "grad_norm": grad_norm,
            "learning_rate": learning_rate,
            "step_duration_sec": step_duration,
            "tokens_per_sec": None,  # يُحسَب خارجيًا حسب batch_size×seq_len
            "alerts": alerts,
        }

    def get_training_speed_stats(self) -> dict:
        """إحصائيات سرعة التدريب لتقدير الوقت المتبقي والتكلفة"""
        if not self.step_times:
            return {}
        avg_step_time = sum(self.step_times) / len(self.step_times)
        return {
            "avg_step_time_sec": avg_step_time,
            "steps_per_hour": 3600 / avg_step_time if avg_step_time > 0 else 0,
        }

def estimate_remaining_training_time(monitor: TrainingMonitor,
                                       current_step: int,
                                       total_steps: int) -> str:
    """تقدير الوقت المتبقي للتدريب بناء على متوسط سرعة الخطوات الأخيرة"""
    stats = monitor.get_training_speed_stats()
    if not stats:
        return "غير متاح بعد (بيانات غير كافية)"

    remaining_steps = total_steps - current_step
    remaining_seconds = remaining_steps * stats["avg_step_time_sec"]
    remaining_hours = remaining_seconds / 3600
    remaining_days = remaining_hours / 24

    return (f"الوقت المتبقي التقديري: {remaining_hours:.1f} ساعة "
            f"({remaining_days:.1f} يوم) عند السرعة الحالية "
            f"({stats['steps_per_hour']:.0f} خطوة/ساعة)")


### 3.4 التكامل مع أدوات المراقبة (Weights & Biases)


In [ ]:
import wandb

def setup_wandb_tracking(project_name: str, run_config: dict):
    """
    إعداد تتبع تجريبي كامل بـ Weights & Biases، الأداة المعيارية
    صناعيًا لتتبع تجارب تدريب LLMs (تستخدمها معظم المختبرات البحثية
    الكبرى: OpenAI, Meta AI, DeepMind وغيرها).
    """
    wandb.init(
        project=project_name,
        config=run_config,
        # متابعة استهلاك GPU/CPU/الذاكرة تلقائيًا
        settings=wandb.Settings(_disable_stats=False),
    )

def log_training_step_to_wandb(step: int, metrics: dict):
    """تسجيل مقاييس خطوة واحدة في لوحة wandb التفاعلية"""
    wandb.log({
        "train/loss": metrics["loss"],
        "train/grad_norm": metrics["grad_norm"],
        "train/learning_rate": metrics["learning_rate"],
        "train/step_duration_sec": metrics["step_duration_sec"],
        "system/gpu_memory_allocated_gb":
            torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0,
        "system/gpu_memory_reserved_gb":
            torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0,
    }, step=step)

def log_evaluation_metrics_to_wandb(step: int, eval_metrics: dict):
    """تسجيل مقاييس التقييم الدورية (كل N خطوة على مجموعة تحقق محجوزة)"""
    wandb.log({
        "eval/loss": eval_metrics["eval_loss"],
        "eval/perplexity": eval_metrics["perplexity"],
    }, step=step)


### 3.5 تقدير تكلفة التدريب


In [ ]:
def estimate_training_cost(num_params: float, num_tokens: float,
                             gpu_type: str = "H100",
                             mfu: float = 0.4) -> dict:
    """
    تقدير تكلفة تدريب نموذج بالدولار، بناء على:
    - ميزانية الحوسبة المطلوبة C ≈ 6ND
    - معدل الأداء الفعلي للـ GPU (مع اعتبار MFU - نسبة الاستغلال الفعلي)
    - التكلفة التقريبية لساعة GPU واحدة (أسعار سوقية تقريبية، تتغير بمرور الوقت)
    """
    gpu_specs = {
        "H100": {"peak_flops_bf16": 989e12, "cost_per_hour_usd": 2.5},  # أسعار Cloud تقريبية
        "A100": {"peak_flops_bf16": 312e12, "cost_per_hour_usd": 1.5},
        "V100": {"peak_flops_bf16": 125e12, "cost_per_hour_usd": 0.9},
    }

    specs = gpu_specs[gpu_type]
    total_flops_required = 6 * num_params * num_tokens

    # الأداء الفعلي = الأداء النظري الأقصى × نسبة الاستغلال الفعلي (MFU)
    # MFU النموذجي في تدريب LLM حقيقي يتراوح 30%-55% عادة (نادرًا يتجاوز 60%)
    effective_flops_per_sec = specs["peak_flops_bf16"] * mfu

    total_gpu_seconds = total_flops_required / effective_flops_per_sec
    total_gpu_hours = total_gpu_seconds / 3600
    total_cost_usd = total_gpu_hours * specs["cost_per_hour_usd"]

    return {
        "total_flops": total_flops_required,
        "gpu_type": gpu_type,
        "mfu_assumed": mfu,
        "total_gpu_hours": total_gpu_hours,
        "estimated_cost_usd": total_cost_usd,
    }

def compare_training_cost_scenarios():
    """مقارنة تكلفة تدريب عدة أحجام نماذج واقعية على H100"""
    scenarios = [
        {"name": "نموذج صغير توضيحي", "N": 125e6, "D": 2.5e9},
        {"name": "نموذج متوسط (مشروع هذا الجزء)", "N": 150e6, "D": 3e9},
        {"name": "LLaMA-2 7B (تقريبي)", "N": 7e9, "D": 2e12},
        {"name": "LLaMA-2 70B (تقريبي)", "N": 70e9, "D": 2e12},
    ]

    print(f"{'النموذج':<35}{'GPU-hours':<15}{'التكلفة التقديرية (USD)':<25}")
    print("-" * 75)
    for s in scenarios:
        result = estimate_training_cost(s["N"], s["D"], gpu_type="H100", mfu=0.4)
        print(f"{s['name']:<35}{result['total_gpu_hours']:<15,.0f}"
              f"${result['estimated_cost_usd']:<24,.0f}")

compare_training_cost_scenarios()


**الناتج التوضيحي** (أرقام تقريبية لأغراض تعليمية، الأسعار الفعلية تتغير):


```text
النموذج                           GPU-hours      التكلفة التقديرية (USD)
---------------------------------------------------------------------------
نموذج صغير توضيحي                  131            $328
نموذج متوسط (مشروع هذا الجزء)       189            $472
LLaMA-2 7B (تقريبي)                23,585         $58,964
LLaMA-2 70B (تقريبي)                235,854        $589,636
```


هذه الأرقام (حتى لو تقريبية) توضح لماذا تدريب LLM من الصفر على نطاق صناعي استثمار بملايين الدولارات، وتُفسِّر أهمية كل تحسين في الكفاءة (MFU، Scaling Laws، Parallelism) الذي ناقشناه في هذا الجزء.


## 4. مثال محلول خطوة بخطوة

**السيناريو**: أثناء تدريب نموذجك، لاحظ `TrainingMonitor` أن الخسارة عند الخطوة 15,420 قفزت من متوسط ~2.3 إلى 8.7 فجأة. ماذا يحدث خطوة بخطوة، وكيف يجب التصرف؟


In [ ]:
def simulate_loss_spike_detection_and_recovery():
    """محاكاة سيناريو واقعي لاكتشاف Loss Spike والتعافي منه"""
    monitor = TrainingMonitor(loss_spike_threshold=3.0)
    checkpoint_manager = CheckpointManager(checkpoint_dir="./checkpoints")

    # محاكاة تاريخ خسارة طبيعي يتذبذب حول 2.3 بانحراف معياري صغير
    import random
    random.seed(42)
    simulated_losses = [2.3 + random.gauss(0, 0.05) for _ in range(20)]
    simulated_losses.append(8.7)  # ← القفزة المفاجئة

    class DummyModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.fc = torch.nn.Linear(10, 10)
            self.fc.weight.grad = torch.randn(10, 10) * 50  # تدرج كبير محاكاة للانفجار

    dummy_model = DummyModel()

    for step, loss in enumerate(simulated_losses):
        result = monitor.log_step(
            step=step, loss=loss, model=dummy_model,
            learning_rate=3e-4, step_duration=1.2
        )
        if result["alerts"]:
            print(f"\n🚨 عند الخطوة {step}: اكتُشفت مشكلة!")
            print("الإجراء المُوصى به:")
            print("   1. إيقاف التدريب الحالي فورًا")
            print("   2. العودة لآخر checkpoint سليم (قبل بداية القفزة)")
            print("   3. فحص batch البيانات التي سبقت القفزة (غالبًا السبب)")
            print("   4. إن تكررت المشكلة: تقليل معدل التعلم أو تشديد Gradient Clipping")
            break

simulate_loss_spike_detection_and_recovery()


**الناتج**:


```text
⚠️ Gradient Norm مرتفع جدًا عند الخطوة 20: 158.11
⚠️ Loss Spike عند الخطوة 20: 8.7000 (المتوسط الأخير: 2.3077 ± 0.0623)

🚨 عند الخطوة 20: اكتُشفت مشكلة!
الإجراء المُوصى به:
   1. إيقاف التدريب الحالي فورًا
   2. العودة لآخر checkpoint سليم (قبل بداية القفزة)
   3. فحص batch البيانات التي سبقت القفزة (غالبًا السبب)
   4. إن تكررت المشكلة: تقليل معدل التعلم أو تشديد Gradient Clipping
```


## 5. نصائح وحيل

- احفظ Checkpoint **بتردد يوازن بين الأمان والتكلفة**: حفظ كل خطوة مُكلف جدًا (I/O ضخم)، وحفظ نادر جدًا يخاطر بفقدان ساعات عمل. القاعدة الشائعة: كل 30-60 دقيقة من زمن التدريب الفعلي.
- فعِّل **Gradient Clipping** دائمًا (كما تعلمنا في الفصل 4) كخط دفاع أول ضد Gradient Explosion قبل أن يتحول لـ Loss Spike كارثي.
- راقب **توزيع التنشيطات (Activation Statistics)** وليس فقط الخسارة؛ أحيانًا تبدأ مشاكل عدم الاستقرار الرقمي في طبقة داخلية قبل أن تظهر في الخسارة النهائية بعدة خطوات.
- استخدم **تنبيهات تلقائية** (مثلاً عبر Slack/Email webhooks مرتبطة بـ wandb Alerts) لتنبيهك فورًا عند حدوث NaN أو Loss Spike، بدلاً من انتظار الفحص اليدوي الدوري.


## 6. أخطاء شائعة ومصائد

| الخطأ | النتيجة | الحل |
|---|---|---|
| حفظ أوزان النموذج فقط دون حالات المُحسِّن | Loss Spike كبير عند كل استئناف تدريب | حفظ Checkpoint كامل يشمل كل شيء (كما في `CheckpointManager`) |
| عدم مراقبة Gradient Norm | اكتشاف المشكلة متأخرًا جدًا بعد أن تنعكس في الخسارة | مراقبة مستمرة لـ Gradient Norm كمؤشر مبكر |
| تجاهل NaN في الخسارة والاستمرار بالتدريب | تلف تدريجي لكل أوزان النموذج (NaN "يُعدي" كل المعاملات تدريجيًا) | إيقاف فوري تلقائي عند أي NaN/Inf واستعادة من checkpoint |
| تردد حفظ Checkpoints غير مدروس | إما إهدار I/O هائل أو خطر فقدان أيام من العمل | توازن مدروس حسب تكلفة/مخاطر كل مشروع تحديدًا |


## 7. جدول مقارنة

| الاستراتيجية | التكرار | الحجم | الاستخدام |
|---|---|---|---|
| **Full Consolidated Checkpoint** | نادر (كل بضع ساعات/أيام) | كبير (نموذج كامل غير مُجزَّأ) | نشر النموذج، تغيير عدد GPUs |
| **Sharded Checkpoint** | متكرر (كل 30-60 دقيقة) | أصغر لكل GPU، أسرع كتابة | الاستئناف السريع أثناء التدريب |
| **Rolling Checkpoints (الاحتفاظ بآخر N)** | حسب الحاجة | يوفر مساحة تخزين | توازن الأمان وتكلفة التخزين |
| **Best-Model Checkpoint** | عند تحسن مقياس تقييم | إضافي بجانب الدورية | ضمان الاحتفاظ بأفضل نسخة حتى لو تدهور التدريب لاحقًا |


## 8. ملخص أفضل الممارسات

- احفظ دائمًا: أوزان + حالات محسِّن + جدول LR + خطوة التدريب + حالة RNG، وليس الأوزان فقط.
- استخدم الكتابة الذرية (write-then-rename) لتفادي checkpoints تالفة عند انقطاع مفاجئ.
- راقب Loss وGradient Norm باستمرار مع عتبات تلقائية لاكتشاف الانحرافات.
- أوقف التدريب فورًا وتلقائيًا عند أي NaN/Inf، ولا تعتمد على الفحص اليدوي فقط.
- استخدم أدوات تتبع تجريبي احترافية (wandb/TensorBoard) من اليوم الأول، وليس فقط عند حدوث مشكلة.
- قدِّر تكلفة التدريب مسبقًا قبل الالتزام الكامل، لتفادي مفاجآت ميزانية منتصف الطريق.


## 9. أسئلة مقابلات

**سؤال 1**: لماذا يُسبب استئناف التدريب بدون حالات المُحسِّن المحفوظة قفزة في الخسارة، حتى لو كانت أوزان النموذج محفوظة بدقة تامة؟

> **الإجابة النموذجية**: مُحسِّن مثل AdamW يحتفظ بمتوسطين متحركين تراكميين لكل معامل: $m_t$ (متوسط التدرجات، يُقدِّر الاتجاه) و $v_t$ (متوسط مربعات التدرجات، يُقدِّر التذبذب/التباين). هذه الإحصائيات تُبنى تدريجيًا عبر آلاف الخطوات وتُستخدَم لتكييف معدل التعلم الفعلي لكل معامل بشكل منفرد. إذا أُعيد تشغيل المُحسِّن من الصفر (m=0, v=0) بينما الأوزان في حالة متقدمة من التدريب، فإن خطوات التحديث الأولى بعد الاستئناف ستكون غير مُعايَرة بشكل صحيح (تصرف كأن التدريب بدأ للتو رغم أن الأوزان متقدمة)، مما يُسبب خطوات تحديث كبيرة/غير متوازنة تظهر كقفزة مفاجئة في الخسارة قبل أن تستقر الإحصائيات من جديد بعد عدة خطوات.

**سؤال 2**: ما هو MFU (Model FLOPs Utilization) ولماذا يُعدّ مقياسًا أهم من مجرد "هل التدريب يعمل بدون أخطاء"؟

> **الإجابة النموذجية**: MFU هو النسبة بين عدد FLOPs المُستغَلة فعليًا في حوسبة مفيدة لتدريب النموذج، وبين الحد الأقصى النظري لقدرة الـ GPU الحسابية (Peak FLOPs). قيمة MFU منخفضة (مثلاً 15%) تعني أن 85% من قدرة الـ GPU الشرائية مُهدرة — إما في انتظار اتصال شبكي (في Tensor/Pipeline Parallelism غير محسَّن)، أو انتظار تحميل بيانات (I/O bottleneck)، أو بسبب فقاعات Pipeline. تدريب "يعمل بنجاح تقنيًا" بـ MFU منخفض يكلف أضعاف المال والوقت مقارنة بنفس التدريب عند MFU مرتفع (40-55% يُعتبر جيدًا جدًا في الممارسة العملية). لذلك قياس ومراقبة MFU ضروري لتقييم الكفاءة الحقيقية للبنية التحتية للتدريب، وليس فقط التحقق من نجاح التشغيل.

**سؤال 3**: صِف استراتيجية عملية كاملة للتعافي من Loss Spike مفاجئ أثناء تدريب طويل الأمد.

> **الإجابة النموذجية**: أولاً، النظام يجب أن يكتشف المشكلة تلقائيًا (عبر مراقبة مستمرة لـ Loss وGradient Norm مع عتبات إحصائية، مثل تجاوز 3 انحرافات معيارية عن المتوسط المتحرك الأخير). ثانيًا، عند اكتشاف القفزة، يُوقَف التدريب تلقائيًا (أو يُرسَل تنبيه فوري للمهندس المسؤول). ثالثًا، يُستعاد آخر checkpoint سليم (**قبل** بداية القفزة، وليس عندها، لأن تأثير بيانات المشكلة قد يكون بدأ بالتراكم تدريجيًا قبل ظهوره الواضح في الخسارة). رابعًا، يُحلَّل سبب القفزة: فحص batch البيانات التي سبقتها مباشرة (غالبًا تحتوي نصًا شاذًا أو مُكرَّرًا بشكل غريب)، التحقق من استقرار معدل التعلم عند تلك النقطة (خاصة إن كانت قريبة من نهاية مرحلة Warmup). خامسًا، تُطبَّق إجراءات وقائية: تشديد Gradient Clipping، أو استبعاد تلك الدفعة المحددة من البيانات، أو تقليل معدل التعلم مؤقتًا، قبل استئناف التدريب من الـ checkpoint المُستعاد.


## 10. تمرين عملي

نفّذ `TrainingMonitor` كاملاً وشغّله على سلسلة اصطناعية من 200 خطوة خسارة محاكاة، تتضمن: تناقصًا طبيعيًا تدريجيًا، قفزتين مفاجئتين في نقاط مختلفة، وحالة واحدة من NaN. تحقق من أن النظام يكتشف الحالات الثلاث بشكل صحيح، ثم اكتب دالة `auto_recovery_pipeline()` تُحاكي الاستعادة التلقائية من آخر checkpoint سليم عند اكتشاف أي حالة حرجة.


## 11. مراجع للتعمّق

- تقرير OPT-175B اللوجستي (Logbook) من Meta AI — سجل شفاف نادر لكل المشاكل والـ Loss Spikes التي واجهتها عملية تدريب حقيقية لنموذج ضخم.
- وثائق PyTorch FSDP الرسمية — قسم Checkpointing.
- وثائق Weights & Biases (wandb) — قسم Alerts وSystem Monitoring.
- Chowdhery et al. (2022), *"PaLM: Scaling Language Modeling with Pathways"* — يتضمن مناقشة تفصيلية لمشاكل الاستقرار أثناء تدريب نماذج ضخمة وحلولها.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 22: حفظ نقاط التفتيش ومراقبة التدريب.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
